# Dự án 3: Phân tích hiệu quả tài chính phim

## Mục tiêu
Bộ dữ liệu hiện có chỉ gồm `budget` và `revenue`, nhưng vẫn đủ để tạo một mini-project
về hiệu quả tài chính:

1. Phim có ngân sách cao có luôn tạo ra doanh thu cao hơn không?
2. Tỷ lệ sinh lời (ROI) phân bố ra sao?
3. Nhóm ngân sách nào có xác suất sinh lời tốt nhất?

## Giới hạn dữ liệu
- Chưa có tên phim, thể loại, ngày phát hành.
- Vì vậy notebook này tập trung vào **phân tích tài chính định lượng**, chưa đi vào storytelling theo từng phim cụ thể.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)

DATA_DIR = Path('data')
movies = pd.read_csv(DATA_DIR / 'financials.csv')
movies = movies.drop(columns=[col for col in movies.columns if col.startswith('Unnamed')], errors='ignore')
movies['profit'] = movies['revenue'] - movies['budget']
movies['roi'] = movies['profit'] / movies['budget']

print('Kích thước dữ liệu:', movies.shape)
movies.head()

## 1. Thống kê tổng quan

In [ ]:
summary = pd.DataFrame({
    'metric': ['so_phim', 'budget_trung_vi', 'revenue_trung_vi', 'profit_share'],
    'value': [
        len(movies),
        movies['budget'].median(),
        movies['revenue'].median(),
        (movies['profit'] > 0).mean()
    ]
})
display(summary)
display(movies[['budget', 'revenue', 'profit', 'roi']].describe().T)

## 2. Quan hệ giữa ngân sách và doanh thu

In [ ]:
ax = movies.plot(kind='scatter', x='budget', y='revenue', alpha=0.5)
ax.set_title('Budget vs revenue')
ax.set_xlabel('Budget (USD)')
ax.set_ylabel('Revenue (USD)')
ax.set_xscale('log')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print('Tương quan budget - revenue:', round(movies['budget'].corr(movies['revenue']), 3))

## 3. ROI và mức độ sinh lời

In [ ]:
roi_trim = movies['roi'].clip(lower=movies['roi'].quantile(0.01), upper=movies['roi'].quantile(0.99))
ax = roi_trim.plot(kind='hist', bins=40)
ax.set_title('ROI distribution (trimmed 1st-99th percentile)')
ax.set_xlabel('ROI')
plt.tight_layout()
plt.show()

## 4. So sánh theo nhóm ngân sách

Chia phim thành 4 nhóm ngân sách để xem nhóm nào có trung vị doanh thu, trung vị lợi nhuận
và tỷ lệ phim có lãi tốt hơn.

In [ ]:
bins = [0, 10_000_000, 50_000_000, 100_000_000, 10_000_000_000]
labels = ['<10M', '$10M-$50M', '$50M-$100M', '>=$100M']
movies['budget_band'] = pd.cut(movies['budget'], bins=bins, labels=labels, include_lowest=True)

band_summary = (
    movies.groupby('budget_band', observed=False)
          .agg(
              so_phim=('id', 'count'),
              budget_trung_vi=('budget', 'median'),
              revenue_trung_vi=('revenue', 'median'),
              profit_trung_vi=('profit', 'median'),
              roi_trung_vi=('roi', 'median'),
              ty_le_co_lai=('profit', lambda s: (s > 0).mean())
          )
          .reset_index()
)
display(band_summary)

In [ ]:
ax = band_summary.plot(kind='bar', x='budget_band', y='roi_trung_vi', legend=False)
ax.set_title('Median ROI by budget band')
ax.set_xlabel('Budget band')
ax.set_ylabel('Median ROI')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
ax = band_summary.plot(kind='bar', x='budget_band', y='ty_le_co_lai', legend=False)
ax.set_title('Share of profitable films by budget band')
ax.set_xlabel('Budget band')
ax.set_ylabel('Profitable share')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Kết luận ngắn

In [ ]:
top_roi = movies[movies['budget'] >= 1_000_000].nlargest(10, 'roi')[['id', 'budget', 'revenue', 'profit', 'roi']]
display(top_roi)

print('Nhận xét:')
print('- Budget và revenue có tương quan dương khá rõ, nghĩa là phim ngân sách cao thường tạo doanh thu cao hơn.')
print('- Tuy vậy, ROI không tăng tuyến tính theo budget. Nhiều phim vốn vừa hoặc thấp vẫn có ROI rất mạnh.')
print('- Nhóm >=$100M có tỷ lệ phim có lãi cao nhất trong bộ dữ liệu này, nhưng nhóm <10M lại có ROI trung vị rất tốt.')
print('- Vì thiếu metadata phim, bước tiếp theo nên là bổ sung title/genre/release_date để kể chuyện tốt hơn trong portfolio.')